In [4]:

import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\MINI PROJECT\venvs\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# %% Cell 1: imports
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


In [6]:
# %% Cell 2: normalization
def normalize(arr):
    return (arr - arr.min()) / (np.ptp(arr) + 1e-8)


In [7]:
# %% Cell 3: user text profile
def build_user_text_vector(user_id, ratings_df, text_embeddings, movie_id_to_index):
    liked_movies = ratings_df[ratings_df.userId == user_id]["movieId"]

    indices = [
        movie_id_to_index[mid]
        for mid in liked_movies if mid in movie_id_to_index
    ]

    if not indices:
        return None

    return text_embeddings[indices].mean(axis=0)


In [8]:
# %% Cell 4: user poster profile
def build_user_poster_vector(user_id, ratings_df, poster_embeddings, movie_id_to_index):
    liked_movies = ratings_df[ratings_df.userId == user_id]["movieId"]

    indices = [
        movie_id_to_index[mid]
        for mid in liked_movies
        if mid in movie_id_to_index and
           np.linalg.norm(poster_embeddings[movie_id_to_index[mid]]) > 0
    ]

    if not indices:
        return None

    return poster_embeddings[indices].mean(axis=0)


In [9]:
# %% Cell 5: like prediction
def predict_user_like_movie(
    user_id,
    movie_id,
    ncf_model,
    ratings_df,
    movies_df,
    text_embeddings,
    poster_embeddings,
    user_encoder,
    movie_encoder,
    alpha=0.5,   # NCF
    beta=0.3,    # Text
    gamma=0.2    # Poster
):
    assert abs(alpha + beta + gamma - 1.0) < 1e-6

    # Encode user & movie
    user_enc = user_encoder.transform([user_id])[0]
    movie_enc = movie_encoder.transform([movie_id])[0]

    # ---------- NCF ----------
    ncf_score = ncf_model.predict(
        [np.array([user_enc]), np.array([movie_enc])],
        verbose=0
    )[0][0]

    # ---------- Content ----------
    movie_id_to_index = dict(zip(movies_df.movieId, movies_df.index))
    idx = movie_id_to_index[movie_id]

    text_score = 0
    poster_score = 0

    user_text_vec = build_user_text_vector(
        user_id, ratings_df, text_embeddings, movie_id_to_index
    )

    if user_text_vec is not None:
        text_score = cosine_similarity(
            user_text_vec.reshape(1, -1),
            text_embeddings[idx].reshape(1, -1)
        )[0][0]

    user_poster_vec = build_user_poster_vector(
        user_id, ratings_df, poster_embeddings, movie_id_to_index
    )

    if user_poster_vec is not None and np.linalg.norm(poster_embeddings[idx]) > 0:
        poster_score = cosine_similarity(
            user_poster_vec.reshape(1, -1),
            poster_embeddings[idx].reshape(1, -1)
        )[0][0]

    # ---------- Normalize ----------
    scores = np.array([ncf_score, text_score, poster_score])
    scores = normalize(scores)

    final_score = (
        alpha * scores[0] +
        beta * scores[1] +
        gamma * scores[2]
    )

    return {
        "final_score": float(final_score),
        "components": {
            "ncf": float(scores[0]),
            "text": float(scores[1]),
            "poster": float(scores[2])
        }
    }


In [10]:
# %% Cell 6: explanation generator
def generate_explanation(user_id, movie_id, result, ratings_df, movies_df):
    title = movies_df[movies_df.movieId == movie_id]["title"].values[0]

    reasons = []

    if result["components"]["ncf"] > 0.4:
        reasons.append(
            "users with viewing patterns similar to yours rated this movie highly"
        )

    if result["components"]["text"] > 0.3:
        reasons.append(
            "the story and genres closely match movies you have enjoyed before"
        )

    if result["components"]["poster"] > 0.3:
        reasons.append(
            "the visual style resembles films you previously liked"
        )

    if not reasons:
        reasons.append(
            "it shares moderate similarities with your past preferences"
        )

    explanation = (
        f"You are likely to enjoy **{title}** because " +
        ", and ".join(reasons) + "."
    )

    return explanation


In [11]:
NCF_MODEL_PATH = r"D:\MINI PROJECT\Checkpoints\ncf_model.keras"
USER_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\user_encoder.pkl"
MOVIE_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\movie_encoder.pkl"
EMBEDDINGS_PATH = r"D:\MINI PROJECT\Checkpoints\movie_embeddings.npy"

MOVIES_PATH = r"D:\MINI PROJECT\DATASET\movies_final.csv"
RATINGS_PATH = r"D:\MINI PROJECT\DATASET\MovieLensDataset20M\rating.csv"

In [12]:
print("Loading data...")
movies = pd.read_csv(MOVIES_PATH)
ratings = pd.read_csv(RATINGS_PATH)

# -------------------------------
# LOAD MODEL & ENCODERS
# -------------------------------
print("Loading NCF model...")
ncfmodel = load_model(NCF_MODEL_PATH)

print("Loading encoders...")
user_encoder = joblib.load(USER_ENCODER_PATH)
movie_encoder = joblib.load(MOVIE_ENCODER_PATH)

text_embeddings=np.load("D:\MINI PROJECT\Checkpoints\movie_embeddings.npy")
poster_embeddings=np.load("D:\MINI PROJECT\Checkpoints\poster_embeddings.npy")


Loading data...
Loading NCF model...
Loading encoders...


In [ ]:
from groq import Groq
import os

import os

api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_llm_explanation(expl_data):
    """
    expl_data: dict with explanation signals
    """

    prompt = f"""
    You are an explainable AI assistant inside a movie recommender system.

    Explain why the movie "{expl_data['recommended_movie']}" was recommended.

    Context:
    - User recently watched: {", ".join(expl_data['recent_movies'])}
    - Movie genres: {expl_data['genres']}
    - Similarity reason: {expl_data['similar_reason']}

    Rules:
    - Natural, friendly language
    - Max 2 sentences
    - No technical terms or model names
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.6,
    )

    return response.choices[0].message.content.strip()


In [21]:
movie_to_test = movies.movieId.iloc[0]

result = predict_user_like_movie(
    user_id=10,
    movie_id=movie_to_test,
    ncf_model=ncfmodel,
    ratings_df=ratings,
    movies_df=movies,
    text_embeddings=text_embeddings,
    poster_embeddings=poster_embeddings,
    user_encoder=user_encoder,
    movie_encoder=movie_encoder
)

# 🔹 FIX: define row
row = movies[movies.movieId == movie_to_test].iloc[0]

recent_titles = (
    movies[movies.movieId.isin(
        ratings[ratings.userId == 10].movieId
    )]["title"].tolist()
)

expl_data = {
    "recommended_movie": row["title"],
    "genres": row["genres"],
    "recent_movies": recent_titles[:3],
    "similar_reason": f"It shares themes with {recent_titles[0]}"
}

explanation = generate_llm_explanation(expl_data)

print("Final score:", result["final_score"])
print("Explanation:", explanation)


Final score: 0.5056091547012329
Explanation: You recently watched Toy Story, and I think I see why I recommended it to you - it shares themes with the movie that you've shown an interest in, like friendship, adventure, and a touch of humor. Since you enjoyed Toy Story, I figured you might also enjoy more films that bring imagination and fun to the big screen.
